In [1]:
!pip install -q git+https://github.com/amazon-science/chronos-forecasting.git
!pip install -q datasets gluonts transformers accelerate scikit-learn
!pip install --upgrade numpy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.1 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1,

In [2]:
import sys
sys.path.insert(0, "/content/chronos-forecasting/src")

import torch
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from chronos import ChronosConfig, ChronosPipeline
from transformers import AutoModelForSeq2SeqLM, Trainer, TrainingArguments
from torch.utils.data import IterableDataset
from datasets import load_dataset
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
import time

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [4]:
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 48
MAX_STEPS = 500
BATCH_SIZE = 8
MAX_TRAIN_SERIES = 200
MAX_TEST_SERIES = 500

DATASET_POOL = {
    "m4_hourly": {"display": "M4 Hourly", "domain": "Economics"},
    "monash_electricity_hourly": {"display": "Electricity", "domain": "Energy"},
    "monash_traffic": {"display": "Traffic", "domain": "Transport"},
    "monash_nn5_weekly": {"display": "NN5 Weekly", "domain": "Retail"},
}

COMBINATIONS = [
    {
        "name": "Economics only",
        "datasets": ["m4_hourly", "monash_electricity_hourly"],
        "desc": "Экономика + Энергетика"
    },
    {
        "name": "Energy + Transport",
        "datasets": ["monash_electricity_hourly", "monash_traffic"],
        "desc": "Энергетика + Транспорт"
    },
    {
        "name": "All three",
        "datasets": ["m4_hourly", "monash_electricity_hourly", "monash_traffic"],
        "desc": "Экономика + Энергетика + Транспорт"
    },
    {
        "name": "Economics + Retail",
        "datasets": ["m4_hourly", "monash_nn5_weekly"],
        "desc": "Экономика + Ритейл"
    },
    {
        "name": "Energy + Retail",
        "datasets": ["monash_electricity_hourly", "monash_nn5_weekly"],
        "desc": "Энергетика + Ритейл"
    },
    {
        "name": "All four",
        "datasets": ["m4_hourly", "monash_electricity_hourly", "monash_traffic", "monash_nn5_weekly"],
        "desc": "Все четыре домена"
    },
]

ALL_TEST_DATASETS = [
    ("m4_yearly", "M4 Yearly", "Economics"),
    ("m4_quarterly", "M4 Quarterly", "Economics"),
    ("m4_monthly", "M4 Monthly", "Economics"),
    ("m4_weekly", "M4 Weekly", "Economics"),
    ("m4_daily", "M4 Daily", "Economics"),
    ("m4_hourly", "M4 Hourly", "Economics"),
    ("monash_electricity_hourly", "Electricity", "Energy"),
    ("monash_traffic", "Traffic", "Transport"),
    ("monash_weather", "Weather", "Meteorology"),
    ("monash_nn5_weekly", "NN5 Weekly", "Retail"),
]

In [5]:
def load_and_prepare_dataset(dataset_name, context_len, pred_len, max_series=None):
    dataset = load_dataset("autogluon/chronos_datasets", dataset_name, split="train")
    df = dataset.to_pandas()
    
    series_ids = df['id'].unique()
    all_series = []
    
    for series_id in series_ids:
        series = df[df['id'] == series_id]['target'].values[0]
        if isinstance(series, np.ndarray):
            series = series.tolist()
        if len(series) > context_len + pred_len + 10:
            series = series[:context_len + pred_len]
            all_series.append(series)
    
    if len(all_series) == 0:
        return [], []
    
    if max_series is not None and len(all_series) > max_series:
        all_series = all_series[:max_series]
    
    train_series, test_series = train_test_split(
        all_series,
        test_size=0.2,
        random_state=42
    )
    
    return train_series, test_series

def prepare_data(series_list, context_len, pred_len):
    data = []
    for series in series_list:
        if len(series) < context_len + pred_len:
            continue
        for i in range(len(series) - context_len - pred_len + 1):
            context = series[i:i+context_len]
            future = series[i+context_len:i+context_len+pred_len]
            if len(context) == context_len and len(future) == pred_len:
                data.append({
                    "context": context,
                    "future": future,
                })
    return data

In [6]:
class ChronosDataset(IterableDataset):
    def __init__(self, data, tokenizer, config):
        self.data = data
        self.tokenizer = tokenizer
        self.vocab_size = config.n_tokens + config.n_special_tokens
        self.indices = list(range(len(data)))

    def __iter__(self):
        np.random.shuffle(self.indices)
        for idx in self.indices:
            item = self.data[idx]
            context = torch.tensor(item["context"], dtype=torch.float32).unsqueeze(0)
            future = torch.tensor(item["future"], dtype=torch.float32).unsqueeze(0)

            input_ids, attention_mask, scale = self.tokenizer.context_input_transform(context)
            labels, _ = self.tokenizer.label_input_transform(future, scale)
            labels = torch.clamp(labels, 0, self.vocab_size - 1)

            yield {
                "input_ids": input_ids.squeeze(0),
                "attention_mask": attention_mask.squeeze(0),
                "labels": labels.squeeze(0),
            }

In [7]:
def evaluate_model(pipeline, test_series, pred_len):
    results = []
    for series in test_series:
        if len(series) < pred_len:
            continue
        context = series[:-pred_len]
        actual = series[-pred_len:]
        
        with torch.no_grad():
            forecast = pipeline.predict(
                torch.tensor(context, dtype=torch.float32),
                prediction_length=pred_len,
                num_samples=20
            )
            forecast_np = forecast.numpy()
            if forecast_np.ndim == 3:
                forecast_np = forecast_np[0]
            median = np.median(forecast_np, axis=0)
        
        results.append({
            'mae': mean_absolute_error(actual, median),
        })
    
    return np.mean([r['mae'] for r in results]) if results else None

def create_combined_dataset(dataset_names, context_len, pred_len, max_series=None):
    all_data = []
    total_series = 0
    for ds_name in dataset_names:
        train_series, _ = load_and_prepare_dataset(
            ds_name, context_len, pred_len, max_series=max_series
        )
        data = prepare_data(train_series, context_len, pred_len)
        all_data.extend(data)
        total_series += len(train_series)
        print(f"      {DATASET_POOL[ds_name]['display']}: {len(train_series)} рядов, {len(data)} примеров")
    
    print(f"      Всего рядов: {total_series}, Всего примеров: {len(all_data)}")
    return all_data

In [8]:
pipeline_zs = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=device,
    torch_dtype=torch.float32,
)

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/185M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

In [9]:
zero_shot_results_all = {}

for ds_name, ds_display, ds_domain in ALL_TEST_DATASETS:
    _, test_series = load_and_prepare_dataset(
        ds_name, CONTEXT_LENGTH, PREDICTION_LENGTH, max_series=MAX_TEST_SERIES
    )
    if test_series:
        avg_mae = evaluate_model(pipeline_zs, test_series, PREDICTION_LENGTH)
        zero_shot_results_all[ds_display] = avg_mae
        print(f"   {ds_display}: {avg_mae:.3f} ({len(test_series)} рядов)")
    else:
        print(f"   {ds_display}: error")

all_combined_results = {}

README.md: 0.00B [00:00, ?B/s]

m4_yearly/train-00000-of-00001.parquet:   0%|          | 0.00/5.49M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/23000 [00:00<?, ? examples/s]

   M4 Yearly: 2394.360 (2 рядов)


m4_quarterly/train-00000-of-00001.parque(…):   0%|          | 0.00/13.3M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/3010 [00:00<?, ? examples/s]

   Weather: 1.980 (100 рядов)


monash_nn5_weekly/train-00000-of-00001.p(…):   0%|          | 0.00/64.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/111 [00:00<?, ? examples/s]

   NN5 Weekly: error


In [11]:
for combo in COMBINATIONS:
    print(f"{combo['name']} ({combo['desc']})")
    
    combined_train_data = create_combined_dataset(
        combo["datasets"],
        CONTEXT_LENGTH, PREDICTION_LENGTH, max_series=MAX_TRAIN_SERIES
    )
    
    model = AutoModelForSeq2SeqLM.from_pretrained(
        "amazon/chronos-t5-small",
        torch_dtype=torch.float32,
    )
    model = model.to(device)
    
    config = ChronosConfig(
        tokenizer_class="MeanScaleUniformBins",
        tokenizer_kwargs={"low_limit": -15.0, "high_limit": 15.0},
        n_tokens=4096,
        pad_token_id=0,
        eos_token_id=1,
        use_eos_token=True,
        model_type="seq2seq",
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        n_special_tokens=2,
        num_samples=20,
        temperature=1.0,
        top_k=50,
        top_p=1.0,
    )
    tokenizer = config.create_tokenizer()
    
    train_dataset = ChronosDataset(combined_train_data, tokenizer, config)
    
    output_dir = f"./finetuned_combined_{combo['name'].replace(' ', '_')}/"
    output_dir_path = Path(output_dir)
    output_dir_path.mkdir(parents=True, exist_ok=True)
    
    training_args = TrainingArguments(
        output_dir=str(output_dir_path),
        per_device_train_batch_size=BATCH_SIZE,
        learning_rate=1e-5,
        max_steps=MAX_STEPS,
        logging_steps=100,
        save_steps=500,
        save_total_limit=2,
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        remove_unused_columns=False,
        dataloader_num_workers=0,
        report_to="none",
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )
    
    trainer.train()
    
    final_checkpoint_dir = output_dir_path / "checkpoint-final"
    trainer.save_model(str(final_checkpoint_dir))
    
    pipeline_ft_combined = ChronosPipeline.from_pretrained(
        str(final_checkpoint_dir),
        device_map=device,
        torch_dtype=torch.float32,
    )
    
    combo_results = {}
    
    for test_ds_name, test_ds_display, test_ds_domain in ALL_TEST_DATASETS:
        _, test_series = load_and_prepare_dataset(
            test_ds_name, CONTEXT_LENGTH, PREDICTION_LENGTH, max_series=MAX_TEST_SERIES
        )
        if test_series:
            avg_mae = evaluate_model(pipeline_ft_combined, test_series, PREDICTION_LENGTH)
            combo_results[test_ds_display] = avg_mae
            zs = zero_shot_results_all.get(test_ds_display, 0)
            imp = ((zs - avg_mae) / zs * 100) if zs > 0 else 0
            print(f"      {test_ds_display}: {avg_mae:.3f} ({imp:+.1f}%)")
        else:
            print(f"      {test_ds_display}: error")
    
    all_combined_results[combo['name']] = combo_results
    
    del model, trainer, pipeline_ft_combined
    torch.cuda.empty_cache()
    time.sleep(2)

Economics only (Экономика + Энергетика)
      M4 Hourly: 160 рядов, 160 примеров
      Electricity: 160 рядов, 160 примеров
      Всего рядов: 320, Всего примеров: 320


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
100,5.913762
200,5.751436
300,5.624487
400,5.574992
500,5.561290


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Step,Training Loss
100,6.603190
200,6.444986
300,6.328651
400,6.268083
500,6.280615


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

      M4 Yearly: 2363.178 (+1.3%)
      M4 Quarterly: 382.701 (+4.0%)
      M4 Monthly: 523.794 (+1.8%)
      M4 Weekly: 502.602 (+0.0%)
      M4 Daily: 266.758 (-0.8%)
      M4 Hourly: 242.182 (-4.4%)
      Electricity: 188.406 (-5.4%)
      Traffic: 0.010 (+21.8%)
      Weather: 1.984 (-0.2%)
      NN5 Weekly: error


In [12]:
all_rows = []
for combo in COMBINATIONS:
    if combo['name'] not in all_combined_results:
        continue
    row = {'Combination': combo['name']}
    for test_ds in zero_shot_results_all.keys():
        val = all_combined_results[combo['name']].get(test_ds, None)
        if val is not None:
            zs = zero_shot_results_all.get(test_ds, 0)
            imp = ((zs - val) / zs * 100) if zs > 0 else 0
            row[test_ds] = f"{val:.3f} ({imp:+.1f}%)"
        else:
            row[test_ds] = "—"
    all_rows.append(row)

comparison_df = pd.DataFrame(all_rows)
print(comparison_df.to_string(index=False))

       Combination        M4 Yearly    M4 Quarterly      M4 Monthly       M4 Weekly        M4 Daily        M4 Hourly     Electricity        Traffic       Weather
    Economics only 2373.597 (+0.9%) 377.984 (+5.2%) 535.539 (-0.4%) 499.644 (+0.6%) 265.979 (-0.5%)  215.333 (+7.2%) 180.781 (-1.2%)  0.013 (-4.6%) 1.985 (-0.2%)
Energy + Transport 2399.515 (-0.2%) 408.443 (-2.5%) 523.298 (+1.9%) 505.881 (-0.6%) 268.081 (-1.3%) 293.296 (-26.4%) 186.216 (-4.2%) 0.010 (+24.7%) 1.985 (-0.2%)
         All three 2363.178 (+1.3%) 382.701 (+4.0%) 523.794 (+1.8%) 502.602 (+0.0%) 266.758 (-0.8%)  242.182 (-4.4%) 188.406 (-5.4%) 0.010 (+21.8%) 1.984 (-0.2%)
Economics + Retail 2378.906 (+0.6%) 393.051 (+1.4%) 533.080 (+0.1%) 501.615 (+0.2%) 266.321 (-0.6%)  222.555 (+4.1%) 186.449 (-4.3%)  0.013 (-1.0%) 1.985 (-0.2%)
   Energy + Retail 2363.714 (+1.3%) 387.777 (+2.7%) 550.566 (-3.2%) 500.018 (+0.5%) 266.650 (-0.7%) 270.968 (-16.8%) 180.654 (-1.1%)  0.013 (-5.0%) 1.984 (-0.2%)
          All four 2363.178 